In [1]:
!pip install qdrant-client sentence-transformers cohere -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.4 MB/s eta 0:00:00


In [2]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer
import cohere

In [3]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
#Let's make a semantic search for Sci-Fi books!
#all-MiniLM-L6-v2 will encode the data you provide.
#Here you will list all the science fiction books in your library. Each book has metadata, a name, author, publication year and a short description.
documents = [
  { "name": "The Time Machine", "description": "A man travels through time and witnesses the evolution of humanity.", "author": "H.G. Wells", "year": 1895 },
  { "name": "Ender's Game", "description": "A young boy is trained to become a military leader in a war against an alien race.", "author": "Orson Scott Card", "year": 1985 },
  { "name": "Brave New World", "description": "A dystopian society where people are genetically engineered and conditioned to conform to a strict social hierarchy.", "author": "Aldous Huxley", "year": 1932 },
  { "name": "The Hitchhiker's Guide to the Galaxy", "description": "A comedic science fiction series following the misadventures of an unwitting human and his alien friend.", "author": "Douglas Adams", "year": 1979 },
  { "name": "Dune", "description": "A desert planet is the site of political intrigue and power struggles.", "author": "Frank Herbert", "year": 1965 },
  { "name": "Foundation", "description": "A mathematician develops a science to predict the future of humanity and works to save civilization from collapse.", "author": "Isaac Asimov", "year": 1951 },
  { "name": "Snow Crash", "description": "A futuristic world where the internet has evolved into a virtual reality metaverse.", "author": "Neal Stephenson", "year": 1992 },
  { " name": "Neuromancer", "description": "A hacker is hired to pull off a near-impossible hack and gets pulled into a web of intrigue.", "author": "William Gibson", "year": 1984 },
  { "name": "The War of the Worlds", "description": "A Martian invasion of Earth throws humanity into chaos.", "author": "H.G. Wells", "year": 1898 },
  { "name": "The Hunger Games", "description": "A dystopian society where teenagers are forced to fight to the death in a televised spectacle.", "author": "Suzanne Collins", "year": 2008 },
  { "name": "The Andromeda Strain", "description": "A deadly virus from outer space threatens to wipe out humanity.", "author": "Michael Crichton", "year": 1969 },
  { "name": "The Left Hand of Darkness", "description": "A human ambassador is sent to a planet where the inhabitants are genderless and can change gender at will.", "author": "Ursula K. Le Guin", "year": 1969 },
  { "name": "The Time Traveler's Wife", "description": "A love story between a man who involuntarily time travels and the woman he loves.", "author": "Audrey Niffenegger", "year": 2003 }
]



In [11]:
#You need to tell Odrant where to store embeddings. This is a basic demo, so your local computer will use its memory as temporary storage.
qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance

In [12]:
qdrant.recreate_collection(
    collection_name="my_books",
    vectors_config=models.VectorParams(
        # size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

/tmp/ipykernel_1357/928735303.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
/tmp/ipykernel_1357/928735303.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

In [19]:
# Let's vectorize descriptions and upload to qdrant
qdrant.upload_points(
    collection_name="my_books",
    points=[
        models.Record(
            id=idx,
            vector=encoder.encode(doc["description"]).tolist(),
            payload=doc
        ) for idx, doc in enumerate(documents)
    ]
)

In [24]:
# Let's now search for something
hits = qdrant.query_points(
    collection_name="my_books",
    query=encoder.encode("Aliens attack our planet").tolist(),
    limit=1
).points
for hit in hits:
  print(hit.payload, "score: ", hit.score)

{'name': 'The War of the Worlds', 'description': 'A Martian invasion of Earth throws humanity into chaos.', 'author': 'H.G. Wells', 'year': 1898} score:  0.526554037546278


In [ ]:
# Finally add the hits as context with prompts and send to Cohere LLM for final answer.
co = cohere.Client("API_KEY")

def generate_text(prompt, temp=0):
  response = co.chat(
      model=" ",
      message=prompt,
      max_tokens=200,
      temperature=temp
  )
  return response.text
for hit in hits:
  context = hit.payload

user_input="Aliens attack our planet?"

prompt = f"""{context}
Given the information above, answer this question: {user_input}"""
print(prompt)
response = generate_text(prompt, temp=0)
print("Answer: ", response)

{'name': 'The War of the Worlds', 'description': 'A Martian invasion of Earth throws humanity into chaos.', 'author': 'H.G. Wells', 'year': 1898}
Given the information above, answer this question: Aliens attack our planet?
Answer:  Yes, according to the description of *The War of the Worlds* by H.G. Wells, the story involves a Martian invasion of Earth, which means aliens (Martians) do attack our planet in the narrative.
